# Поиск полостей: полный пример

Сетка + вершины Вороного → объединение близких точек → симметрия → геометрия.
Затем две независимые ветки: **координация** и **Li–Li**, а также обе последовательности этих фильтров.

Семь CIF находятся в `inputs/`. Координаты кандидатов на этапах отбора не меняются.
Геометрический допуск — **10% радиуса пробного иона**; углы сравниваются по парам элементов.
Результаты сохраняются в `voids_gallery/`, повторный запуск — в новую папку. Визуализация: CIF для VESTA и POSCAR (`.vasp`) с маркерами X; это файлы просмотра, не готовые входы расчётов VASP.

Зависимости: пакет из корня репозитория (`elec_voids_search/requirements.txt`), Jupyter и IPython. Выполните ячейки сверху вниз.

**Обновление координации:** повторно рассчитан только координационный фильтр на сохранённой геометрической выборке; Li–Li взят из полного прогона. Итоговые таблицы и файлы 07/09 обновлены. Первая сфера должна совпадать по составу и отдельно пройти допуски длин связей и углов.

In [1]:
from pathlib import Path
from dataclasses import replace
from datetime import datetime, timezone
from contextlib import redirect_stdout, redirect_stderr
from time import perf_counter
import hashlib, importlib.metadata, io, json, sys, warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from pymatgen.core import Structure, DummySpecies
from pymatgen.io.vasp import Poscar
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "elec_voids_search/config.py").is_file())
sys.path.insert(0, str(ROOT))
from elec_voids_search.config import SearchConfig
from elec_voids_search.post_geometry_config import PostGeometryConfig
from elec_voids_search.screening_pipeline import run_screening
from elec_voids_search.post_geometry_exports import write_candidate_markers

EXAMPLE = ROOT / "examples/full_pipeline"
OUTPUT = EXAMPLE / "voids_gallery"
if OUTPUT.exists():
    OUTPUT = EXAMPLE / ("voids_gallery_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))

## Настройки

Степени окисления задаются вручную. Все остальные параметры доступны через `SearchConfig`, `VoronoiCoordinationConfig` и `ProbeSeparationConfig`; ниже выводятся фактически используемые значения. `geometry_minimum_fit_fraction=1.0` оставляем неизменным, смягчение регулируем процентным допуском.

Первая сфера: `first_shell_distance_ratio=1.25`, `first_shell_radial_tolerance_A=0.15`, `first_shell_angular_tolerance_deg=12.0`. Состав по элементам должен совпадать точно. `maximum_primary_score=1.0`.

In [1]:
SEARCH = SearchConfig(geometry_overlap_tolerance_percent=10.0)
POST = PostGeometryConfig(paths={
    "coordination": ("coordination",),
    "li_distance": ("li_distance",),
    "coord_then_li": ("coordination", "li_distance"),
    "li_then_coord": ("li_distance", "coordination"),
})
CHARGES = {
    "LATP_sc_integer152_stoichiometric": {"Li": 1, "Ti": 4, "Al": 3, "P": 5, "O": -2},
    "LLZO_cub_ga": {"Li": 1, "La": 3, "Zr": 4, "Ga": 3, "O": -2},
    "Li10Ge(PS6)2": {"Li": 1, "Ge": 4, "P": 5, "S": -2},
    "Li3PO4": {"Li": 1, "P": 5, "O": -2},
    "LiCl": {"Li": 1, "Cl": -1},
    "LiCl_one_Li_vacancy": {"Li": 1, "Cl": -1},
    "Li3PO4_one_Li_vacancy": {"Li": 1, "P": 5, "O": -2},
}
display(pd.Series(SEARCH.to_dict(), name="Поиск и геометрия").to_frame())
display(pd.json_normalize(POST.to_dict(), sep=".").T.rename(columns={0: "После геометрии"}))

,Поиск и геометрия
grid_resolution_A,0.2
grid_search_cutoff_A,5.0
grid_symprec_A,0.1
grid_minima_window,3
grid_r_min_A,1.0
grid_empty_framework,False
grid_verbose,False
probe_element,Li
probe_oxidation_state,1
voronoi_image_range,1


,После геометрии
probe_element,Li
probe_oxidation_state,1
export_cif,False
export_rejected_cif,True
coordination.initial_cutoff_A,6.0
coordination.maximum_cutoff_A,24.0
coordination.cutoff_growth,1.5
coordination.minimum_face_area_A2,0.0
coordination.solid_angle_sum_tolerance,0.00001
coordination.significant_face_fraction,0.01


## Полный расчёт и сохранение

Для каждой структуры сохраняются одна общая таблица координат со статусами этапов, радиусы и занятые контрольные сайты. В `cif/` и `vasp/` — каркас с X-маркерами после каждого этапа. Два порядка последовательных фильтров дают одно пересечение и используют общий файл `09_both`.

Сырые кандидаты сетки и Вороного независимы: их количества не образуют последовательность. После объединения вдоль каждой ветки количество только уменьшается.

In [3]:
OUTPUT.mkdir(parents=True, exist_ok=False)
results, summary_rows, branch_rows, run_notes = {}, [], [], {}
source_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (ROOT / "elec_voids_search").glob("*.py")}

for name, charges in CHARGES.items():
    start = perf_counter()
    path = EXAMPLE / "inputs" / f"{name}.cif"
    settings = replace(SEARCH, geometry_oxidation_states=charges)
    captured = io.StringIO()
    with redirect_stdout(captured), redirect_stderr(captured), warnings.catch_warnings(record=True) as notes:
        warnings.simplefilter("always")
        result = run_screening(path, search_config=settings, post_config=POST)
        host = Structure.from_file(path)
        search, post = result["search"], result["post_geometry"]
        branches = post["branches"]
        stages = {"01_grid": search["grid"], "02_voronoi": search["voronoi"],
                  "03_combined": search["combined"], "04_merged": search["merged"],
                  "05_symmetry": search["nonequivalent"], "06_geometry": search["geometry"],
                  "07_coordination": branches["coordination"]["passed"],
                  "08_li_distance": branches["li_distance"]["passed"],
                  "09_both": branches["coord_then_li"]["passed"]}
        folder = OUTPUT / name
        for fmt in ("cif", "vasp"):
            (folder / fmt).mkdir(parents=True, exist_ok=False)
        for stage, table in stages.items():
            write_candidate_markers(host, table, folder / "cif" / f"{stage}.cif")
            marked = host.copy()
            for point in table[["frac_x", "frac_y", "frac_z"]].to_numpy(dtype=float):
                marked.append(DummySpecies("X"), point)
            Poscar(marked, comment=f"{name}: {stage}; X = candidate markers").write_file(folder / "vasp" / f"{stage}.vasp")
        # Одна таблица вместо повторных выгрузок координат для каждого этапа.
        candidates = search["combined"].drop(columns=["merge_member_candidate_ids", "merge_source_methods"], errors="ignore").copy()
        for stage, table in stages.items():
            candidates[f"in_{stage[3:]}"] = candidates.candidate_id.isin(table.candidate_id)
        for table in (search["geometry_evaluated"], post["candidates"]):
            fields = ["candidate_id"] + [c for c in table if c not in candidates.columns]
            candidates = candidates.merge(table[fields], on="candidate_id", how="left", validate="one_to_one")
        candidates.to_csv(folder / "candidates.csv", index=False)
        search["site_radii"].to_csv(folder / "site_radii.csv", index=False)
        search["occupied_probe_controls"].to_csv(folder / "occupied_geometry_controls.csv", index=False)
        if "occupied_reference_descriptors" in post:
            post["occupied_reference_descriptors"].to_csv(folder / "occupied_descriptors.csv", index=False)
        rows = post["summary"].copy()
        rows.insert(0, "structure", name)
        branch_rows.extend(rows.to_dict("records"))
        row = {"Соединение": name, "Сетка": len(search["grid"]), "Вороной": len(search["voronoi"]),
               "Объединено": len(search["combined"]), "Кластеры": len(search["merged"]),
               "Симметрия": len(search["nonequivalent"]), "Геометрия": len(search["geometry"]),
               "Координация": len(branches["coordination"]["passed"]),
               "Li–Li": len(branches["li_distance"]["passed"]),
               "Коорд. → Li–Li": len(branches["coord_then_li"]["passed"]),
               "Li–Li → коорд.": len(branches["li_then_coord"]["passed"])}
        summary_rows.append(row)
    results[name] = result
    run_notes[name] = {"seconds": round(perf_counter()-start, 2), "settings": settings.to_dict(),
                       "input_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                       "warnings": sorted(set(str(w.message) for w in notes))}
    print(f"{name}: геометрия {row['Геометрия']}, координация {row['Координация']}, Li–Li {row['Li–Li']}, вместе {row['Коорд. → Li–Li']}", flush=True)

summary = pd.DataFrame(summary_rows)
branch_steps = pd.DataFrame(branch_rows)
summary.to_csv(OUTPUT / "summary.csv", index=False)
branch_steps.to_csv(OUTPUT / "branch_steps.csv", index=False)
metadata = {"created_utc": datetime.now(timezone.utc).isoformat(), "search_source_sha256": source_hashes,
            "post_geometry_settings": POST.to_dict(), "structures": run_notes,
            "versions": {name: importlib.metadata.version(name) for name in ("numpy", "scipy", "pandas", "pymatgen", "gridmlip")}}
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\n")
print(f"Сохранено: {OUTPUT.relative_to(ROOT)}")

Полный поиск выполнен ранее. Координация обновлена на сохранённой геометрии; Li–Li не пересчитывался. Актуальные результаты приведены ниже.


## Итоговые количества

«Координация» и «Li–Li» считаются независимо от геометрического входа. Последние столбцы — последовательное применение обоих фильтров. Таблицы ниже показывают каждый шаг последовательностей отдельно, включая отклонённые и неопределённые результаты.

In [2]:
display(summary)
for path, title in [("coordination", "Координация"), ("li_distance", "Li–Li"),
                    ("coord_then_li", "Координация → Li–Li"), ("li_then_coord", "Li–Li → координация")]:
    display(Markdown(f"**{title}**"))
    display(branch_steps.loc[branch_steps.path.eq(path)].drop(columns="path").reset_index(drop=True))

,Соединение,Сетка,Вороной,Объединено,Кластеры,Симметрия,Геометрия,Координация,Li–Li,Коорд. → Li–Li,Li–Li → коорд.
0,LATP_sc_integer152_stoichiometric,278,936,1214,170,88,46,3,3,3,3
1,LLZO_cub_ga,519,1271,1790,320,320,35,16,17,1,1
2,Li10Ge(PS6)2,76,330,406,136,25,5,0,3,0,0
3,Li3PO4,24,116,140,14,5,0,0,0,0,0
4,LiCl,16,10,26,6,3,1,0,0,0,0
5,LiCl_one_Li_vacancy,19,11,30,8,6,3,1,1,1,1
6,Li3PO4_one_Li_vacancy,26,104,130,17,13,4,1,1,1,1


**Координация**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,coordination,46,3,43,0
1,LLZO_cub_ga,1,coordination,35,16,19,0
2,Li10Ge(PS6)2,1,coordination,5,0,5,0
3,Li3PO4,1,coordination,0,0,0,0
4,LiCl,1,coordination,1,0,1,0
5,LiCl_one_Li_vacancy,1,coordination,3,1,2,0
6,Li3PO4_one_Li_vacancy,1,coordination,4,1,3,0


**Li–Li**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,li_distance,46,3,43,0
1,LLZO_cub_ga,1,li_distance,35,17,18,0
2,Li10Ge(PS6)2,1,li_distance,5,3,2,0
3,Li3PO4,1,li_distance,0,0,0,0
4,LiCl,1,li_distance,1,0,1,0
5,LiCl_one_Li_vacancy,1,li_distance,3,1,2,0
6,Li3PO4_one_Li_vacancy,1,li_distance,4,1,3,0


**Координация → Li–Li**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,coordination,46,3,43,0
1,LATP_sc_integer152_stoichiometric,2,li_distance,3,3,0,0
2,LLZO_cub_ga,1,coordination,35,16,19,0
3,LLZO_cub_ga,2,li_distance,16,1,15,0
4,Li10Ge(PS6)2,1,coordination,5,0,5,0
5,Li10Ge(PS6)2,2,li_distance,0,0,0,0
6,Li3PO4,1,coordination,0,0,0,0
7,Li3PO4,2,li_distance,0,0,0,0
8,LiCl,1,coordination,1,0,1,0
9,LiCl,2,li_distance,0,0,0,0


**Li–Li → координация**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,li_distance,46,3,43,0
1,LATP_sc_integer152_stoichiometric,2,coordination,3,3,0,0
2,LLZO_cub_ga,1,li_distance,35,17,18,0
3,LLZO_cub_ga,2,coordination,17,1,16,0
4,Li10Ge(PS6)2,1,li_distance,5,3,2,0
5,Li10Ge(PS6)2,2,coordination,3,0,3,0
6,Li3PO4,1,li_distance,0,0,0,0
7,Li3PO4,2,coordination,0,0,0,0
8,LiCl,1,li_distance,1,0,1,0
9,LiCl,2,coordination,0,0,0,0


## Контроль известных пустых позиций

Для двух искусственных вакансий и шести LATP-маркеров приводится минимальное периодическое расстояние до ближайшего кандидата с учётом симметрийных образов эталона. Условие восстановления позиции — ≤ 0,25 Å; это диагностический допуск, он не участвует в отборе.

In [3]:
targets = json.loads((EXAMPLE / "inputs/vacancy_targets.json").read_text())
latp = pd.read_csv(EXAMPLE / "inputs/LATP_reference_sites.csv")
checks = [(name, "removed_Li", point) for name, point in targets.items()]
checks += [("LATP_sc_integer152_stoichiometric", f"marker_{int(row.source_atom_number_1based)}",
            [row.frac_x, row.frac_y, row.frac_z]) for _, row in latp.loc[latp.resolved_state.eq("intentionally_empty")].iterrows()]
recovery_rows = []
for name, target, point in checks:
    host = Structure.from_file(EXAMPLE / "inputs" / f"{name}.cif")
    search, post = results[name]["search"], results[name]["post_geometry"]
    operations = SpacegroupAnalyzer(host, symprec=SEARCH.symmetry_symprec_A,
                                   angle_tolerance=SEARCH.symmetry_angle_tolerance_deg).get_symmetry_operations()
    orbit = [operation.operate(point) % 1 for operation in operations]
    stages = {"Сетка": search["grid"], "Вороной": search["voronoi"], "Объединено": search["combined"],
              "Кластеры": search["merged"], "Симметрия": search["nonequivalent"], "Геометрия": search["geometry"]}
    stages.update({key: branch["passed"] for key, branch in post["branches"].items()})
    for stage, table in stages.items():
        error = float(host.lattice.get_all_distances(orbit, table[["frac_x", "frac_y", "frac_z"]].to_numpy(dtype=float)).min()) if len(table) else np.nan
        recovery_rows.append({"structure": name, "target": target, "stage": stage,
                              "error_A": error, "recovered": bool(np.isfinite(error) and error <= .25)})
recovery = pd.DataFrame(recovery_rows)
recovery.to_csv(OUTPUT / "reference_recovery.csv", index=False)
display(recovery.pivot(index=["structure", "target"], columns="stage", values="recovered"))
display(recovery.loc[recovery.stage.isin(["Геометрия", "coordination", "li_distance", "coord_then_li"])].reset_index(drop=True))

stage                                         coord_then_li  ...  Симметрия
structure                         target                     ...           
LATP_sc_integer152_stoichiometric marker_146           True  ...       True
                                  marker_147           True  ...       True
                                  marker_148           True  ...       True
                                  marker_149           True  ...       True
                                  marker_150           True  ...       True
                                  marker_152           True  ...       True
Li3PO4_one_Li_vacancy             removed_Li           True  ...       True
LiCl_one_Li_vacancy               removed_Li           True  ...       True

[8 rows x 10 columns]

,structure,target,stage,error_A,recovered
0,LiCl_one_Li_vacancy,removed_Li,Геометрия,1.655591e-04,True
1,LiCl_one_Li_vacancy,removed_Li,coordination,1.655591e-04,True
2,LiCl_one_Li_vacancy,removed_Li,li_distance,1.655591e-04,True
3,LiCl_one_Li_vacancy,removed_Li,coord_then_li,1.655591e-04,True
4,Li3PO4_one_Li_vacancy,removed_Li,Геометрия,4.215082e-02,True
5,Li3PO4_one_Li_vacancy,removed_Li,coordination,4.215082e-02,True
6,Li3PO4_one_Li_vacancy,removed_Li,li_distance,4.215082e-02,True
7,Li3PO4_one_Li_vacancy,removed_Li,coord_then_li,4.215082e-02,True
8,LATP_sc_integer152_stoichiometric,marker_146,Геометрия,8.078836e-08,True
9,LATP_sc_integer152_stoichiometric,marker_146,coordination,8.078836e-08,True


## Результат контрольного прогона

В LATP найдены **3 симметрично неэквивалентных представителя**, соответствующие всем **6 известным пустым маркерам**; они проходят оба фильтра. Обе искусственные вакансии также сохранены: ошибка положения LiCl — **0,000166 Å**, Li₃PO₄ — **0,042151 Å**. Оба порядка применения финальных фильтров дают одинаковое пересечение.

Сырые точки могут совпадать до кластеризации. Для таких CIF некоторые библиотеки сообщают превышение заселённости при объединении сайтов; POSCAR сохраняет список маркеров напрямую. Файлы X предназначены для просмотра.

Эти контроли подтверждают восстановление известных позиций; энергетическая устойчивость и точность на остальных кандидатах требуют отдельной проверки.

После обязательной проверки первой сферы для Li₃PO₄ с вакансией координацию проходит только кандидат **122** (O₄). В LGPS при текущих критериях совпадений нет; отсутствие совпадения не доказывает отсутствие физических позиций внедрения.